# TP3 Application de la loi des grand nombres: estimation de la fiabilit? d'une estimation par bootstrap

On ne peut pas ma?triser les sujets techniques sans pratique, c'est pourquoi les travaux pratiques (TPs) sont une partie importante du cours de sciences des donn?es.

Si vous restez bloqu? plus de quelques minutes sur des questions dans une partie non not?e d'un TP, demandez de l'aide ? un enseignant ou ? un camarade de classe (Expliquer les choses est ?galement b?n?fique : la meilleure fa?on de consolider votre connaissance d'un sujet est de l'expliquer). Ne vous contentez pas de juste partager vos r?ponses sans explication, personne n'en b?n?ficierait.


#### TP d'aujourd'hui

Dans le TP d'aujourd'hui, vous allez travailler les comp?tences suivantes:

1. Impl?menter la m?thode du bootstrap en python
2. Appliquer la m?thode du bootstrap pour ?valuer l'incertitude associ?e ? des estimations
 
<font color='red'>**La deuxi?me partie du TP est ? rendre et not?e. Le rendu doit obligatoirement se faire par le biais d'am?tice avant la fin de la s?ance de TP.**</font>

# 1. Evaluation de l'incertitude associ?e ? une estimation par la m?thode du boostrap (dur?e indicative 1h)

## 1.1 Impl?mentation de la m?thode du bootstrap en python (dur?e indicative 30min)

Nous commencons par importer les modules numpy, pandas et seaborn dont nous allons avoir besoin

In [ ]:
import numpy as np
import pandas
import seaborn

Nous cr?ons un g?n?rateur de nombre al?atoires ? l'aide de la librairie numpy.random, que nous appelons `rng`.

In [ ]:
rng = np.random.default_rng()

A pr?sent, impl?mentons la m?thode du bootstrap en python. Pour rappel, ?tant donn? un ?chantillon de donn?es $x_1, x_2, ..., x_n$ et une estimation $\hat{\theta} = f(x_1, x_2, ..., x_n)$ sur la base de cet ?chantillon d'un param?tre d'int?r?t $\theta$ (par exemple pour estimer la moyenne de la population dont est tir? l'?chantillon, on pourrait calculer la moyenne de l'?chantillon), la m?thode consiste ?:

- Tirer un grand nombre d'?chantillons de taille $n$ de la distribution empirique de l'?chantillon. Disons $1000$ pour ce TP
- Estimer le param?tre d'int?r?t s?par?mment pour chacun de ces $1000$ ?chantillons de taille
   $n$. Notons $\tilde{\theta}_1, \tilde{\theta}_2,..., \tilde{\theta}_{1000}$ les $1000$ valuers obtenues
- Se baser sur la distribution empirique de $\tilde{\theta}_1, \tilde{\theta}_2,..., \tilde{\theta}_{1000}$ pour ?valuer l'incertitude associ?e ? notre estimation $\hat{\theta}$ de $\theta$.

Pour impl?menter cette m?thode en python, commen?ons par ?crire une fonction `get_samples_from_empirical_distribution_of_base_sample` pour tirer des ?chantillons de la distribution empirique de notre ?chantillon de d?part $x_1, x_2, ..., x_n$.

Nous ?crivons une fonction qui prend en argument l'?chantillon de d?part sous le nom  `base_sample` et le nombre d'?chantillons ? tirer sous le nom `nb_samples` et qui renvoie l'?chantillon obtenu sous la forme d'une liste.

Tirer un ?chantillon de la distribution empirique de l'?chantillon $x_1, x_2, ..., x_n$ revient simplement ? choisir un ?l?ment de l'?chantillon au hasard, ce qui peut ?tre impl?ment? avec la m?thode `choice` des g?n?rateurs de nombre al?atoire de `numpy.random`.

In [ ]:
def get_samples_from_empirical_distribution_of_base_sample(base_sample, nb_samples):
    samples = []
    for i in range(nb_samples):
        samples.append(rng.choice(base_sample))
    return samples

Testons notre m?thode sur un ?chantillon invent? $2, 2, 3, 4$. La distribution empirique de cette ?chantillon donne une probabilit? de 1/2 ? la valeur 2, et de 1/4 aux valeurs 3 et 4.

In [ ]:
get_samples_from_empirical_distribution_of_base_sample([2, 2, 3, 4], 10)

Les valeurs renvoy?e par la fonction semble bien respecter les proportions dict?es par la distribution empirique de l'?chantillon de d?part

A pr?sent, impl?mentons une fonction `reestimate_parameter` qui tire un ?chantillon de la distribution empirique de l'?chantillon de d?part de m?me taille que l'?chantillon de d?part (ce genre d'?chantillon est souvent appel? un r?-?chantillon de l'?chantillon de d?part) et qui estime le param?tre d'int?r?t sur la base de ce r?-?chantillon.

Nous ?crivons une fonction qui prend en argument l'?chantillon de d?part sous le nom `base_sample` et la fonction utilis?e pour estimer le param?tre d'int?r?t sous le nom `estimation_function` (si on veut estimer la moyenne d'une population ? partir d'un ?chantillon de cette population, on pourrait par exemple donner comme valeur ? cet argument la fonction `np.mean`).

La fonction appelle la fonction `get_samples_from_empirical_distribution_of_base_sample` cod?e pr?c?demment pour obtenir le r?-?chantillon et retourne le r?sultat de l'application de la fonction `estimation_function` ? ce r?-?chantillon.

In [ ]:
def reestimate_parameter(base_sample, estimation_function):
    resample = get_samples_from_empirical_distribution_of_base_sample(base_sample, len(base_sample))
    return estimation_function(resample)

Testons notre fonction sur notre ?chantillon invent? $2, 2, 3, 4$ en utilisant comme fonction d'estimation la moyenne `np.mean`. La moyenne de l'?chantillon est $11/4=2.75$. La moyenne des r?-?chantillons va d?pendre des valeurs incluse dans le r?-?chantillon et devrait fluctuer entre un minimum de $2$ pour un r?-?chantillon compos? de quatre $2$ et un maximum de $4$ pour un r?-?chantillon compos? de quatre $4$.

In [ ]:
reestimate_parameter(base_sample=[2, 2, 3, 4], estimation_function=np.mean)

Pour finir d'impl?menter la m?thode du bootstrap telle que d?crite plus haut, il ne nous reste plus qu'? r?iterer $1000$ fois le processus de r?-estimation du param?tre d'int?r?t ? partir d'un r?-?chantillon.

In [ ]:
def get_boostrap_distribution_of_estimated_parameter(base_sample, estimation_function, nb_resamples=1000):
    samples = []
    for i in range(nb_resamples):
        samples.append(reestimate_parameter(base_sample, estimation_function))
    return samples

Testons ? pr?sent la m?thode compl?te sur un ?chantillon de taile $400$ d'une distribution uniforme sur l'intervalle $[0,1[$ pour l'estimation de la moyenne. Ici, comme il s'agit d'un ?chantillon simul?, nous connaissons sa distribution r?elle (distribution uniforme sur $[0,1[$) et nous savons donc que la moyenne de la population est l'esp?rance de cette distribution, c'est ? dire $1/2$ (dans le reste du TP nous travaillerons avec des donn?es r?elles, pour lesquelles la distribution de la population n'est pas connue).

Cr?ons d'abord l'?chantillon d?sir? et repr?sentons sa distribution empirique par le biais d'un histogramme

In [ ]:
base_sample = rng.uniform(0, 1, 400)
seaborn.displot(data=base_sample, kind='hist', stat='density')

La taille de l'?chantillon semble suffisant ici pour que la distribution empirique de l'?chantillon fournisse une approximation raisonnable de la distribution r?elle des donn?es. On s'attend donc ? ce que la m?thode du bootstrap, qui d?pend de la qualit? de cette approximation, donne des r?sultats fiables ici.

In [ ]:
np.mean(base_sample)

Nous voyons aussi ci-dessus que la moyenne de l'?chantillon est assez proche de la moyenne de la population.

Appliquons la m?thode du boostrap ? notre exemple

In [ ]:
estimation_function = np.mean
mean_estimates_from_boostrap_method = get_boostrap_distribution_of_estimated_parameter(base_sample, estimation_function)

Repr?sentons sur un histogramme la distribution produite par la m?thode du boostrap pour notre estimation de la moyenne

In [ ]:
seaborn.displot(data=mean_estimates_from_boostrap_method , kind='hist', stat='density')

Comparons avec la distribution r?elle pour notre estimation de la moyenne, qu'on peut obtenir un tirant un grand nombre de fois des ?chantillons de taille 400 de la vraie distribution des donn?es

In [ ]:
actual_mean_estimates = []
for i in range(1000):
    sample = rng.uniform(0, 1, 400)
    actual_mean_estimates.append(np.mean(sample))
seaborn.displot(data=actual_mean_estimates, kind='hist', stat='density')

Nous pouvons aussi calculer les ?carts-types pour la distribution obtenue par boostrap et pour la distribution r?elle de notre estimateur de la moyenne d'une loi uniforme ? partir d'un ?chantillon de taille $400$:

In [ ]:
print(np.array(actual_mean_estimates).std())
print(np.array(mean_estimates_from_boostrap_method).std())

On voit que la m?thode du boostrap repr?sente tr?s bien l'incertitude associ?e ? l'estimation de la moyenne d'une loi uniforme ? partir d'un ?chantillon de taille $400$, avec peut-?tre une l?g?re sous-estimation de cette incertitude.

Ce qui est remarquable c'est qu'elle permet de faire cela sur la base d'un seul ?chantillon de taille $400$.

## 1.2 Application au jeu de donn?es sur les salaires en science des donn?es (dur?e indicative 30min)

(Jeu de donn?es utilis? pr?c?demment dans le cadre du TP1)

In [ ]:
import pandas

data = pandas.read_csv("ds_salaries.csv")
data

Ici nous voudrions savoir si la taille des entreprise influe sur le salaire. Nous allons nous concentrer sur un sous-ensemble du jeu de donn?es pour essayer d'aborder cette question en minimisant l'influence de variables confondantes. Nous choisissons de nous concentrer sur les employ?s seniors, travaillant ? temps plein aux Etats-Unis dans une entreprise des Etats-Unis, qui constituent une grande part du jeu de donn?es.

Regardons le nombre de points de donn?es disponibles par ann?e et taille d'entreprise:

In [ ]:
data[(data['experience_level'] == 'SE') &
     (data['employment_type'] == 'FT') &
     (data['employee_residence'] == 'US') &
     (data['company_location'] == 'US')].groupby(['company_size', 'work_year'])['salary_in_usd'].count()

Il semble y avoir suffisamment de donn?es pour esp?rer obtenir des r?sultats int?ressants en 2022 et 2023. Regardons ce qui se passe en 2022 (vous pouvez refaire l'analyse pour 2023 en changeant la valeur de la variable ci-dessous)

In [ ]:
year=2022 # 2023

Ci-dessous nous s?lectionnons la portion des donn?es qui nous int?resse et l'assemblons dans un dataframe `df_LM`

In [ ]:
df_L = data[(data['experience_level'] == 'SE') &
                (data['employment_type'] == 'FT') &
                (data['employee_residence'] == 'US') &
                (data['company_location'] == 'US') & 
                (data['work_year'] == year) &
                (data['company_size'] == 'L')]

In [ ]:
df_M = data[(data['experience_level'] == 'SE') &
                (data['employment_type'] == 'FT') &
                (data['employee_residence'] == 'US') &
                (data['company_location'] == 'US') & 
                (data['work_year'] == year) &
                (data['company_size'] == 'M')]

In [ ]:
df_LM = pandas.concat([df_L, df_M])[['company_size', 'salary_in_usd']]


In [ ]:
df_LM

Regardons les salaires moyens pour les moyennes et grandes entreprises

In [ ]:
df_LM.groupby('company_size').mean()

Le salaire semble plus ?lev? dans les grandes entreprises. Mais la diff?rence observ?e pourrait-elle ?tre due ? des fluctuations al?atoires ou repr?sente-t-elle une diff?rence r?elle de salaire entre les deux types d'entreprises?

Nous pouvons regarder la distribution empirique des salaires dans notre ?chantillon

In [ ]:
seaborn.displot(data=df_LM, x='salary_in_usd', hue='company_size', kind='hist', stat='density', common_norm=False)

La diff?rence n'est pas flagrante sur cette figure... Utilisons la m?thode du bootstrap pour essayer d'y voir plus clair

In [ ]:
sample_L = df_L['salary_in_usd']
sample_M = df_M['salary_in_usd']

In [ ]:
sample_L

In [ ]:
sample_M

In [ ]:
bootstrapped_mean_salary_L = get_boostrap_distribution_of_estimated_parameter(sample_L, np.mean)
bootstrapped_mean_salary_M = get_boostrap_distribution_of_estimated_parameter(sample_M, np.mean)

Mettons les r?sultats dans une DataFrame pandas pour pouvoir les repr?senter graphiquement avec seaborn

In [ ]:
bL = pandas.DataFrame({'mean_salary': bootstrapped_mean_salary_L, 'size': ['L']*len(bootstrapped_mean_salary_L)})
bM = pandas.DataFrame({'mean_salary': bootstrapped_mean_salary_M, 'size': ['M']*len(bootstrapped_mean_salary_M)})
b = pandas.concat([bL, bM])

In [ ]:
seaborn.displot(data=b, x='mean_salary', hue='size', kind='hist', stat='density', common_norm=False)

Les distributions estim?es par bootstrap pour le salaires moyen dans les deux types d'entreprises sont relativement bien s?par?s. Cela sugg?re que la diff?rence de salaire observ?e n'est pas une simple fluctuation al?atoire.

Cela reste un r?sultat tr?s pr?liminaire cependant. Nos conclusions pourraient ?tre beaucoup plus fortes si nous avions une hypoth?se causale concernant l'origine de cette diff?rence de salaires et si nous r?alisions un exp?rience soigneusement con?ue pour tester cette hypoth?se causale.

# 2. G?n?ralisation ? un nouvel exemple <font color='red'>(partie not?e, ? rendre individuellement, dur?e indicative 1h)</font>

<font color='red'>**Cette partie est ? rendre et not?e. Le rendu doit obligatoirement se faire par le biais d'am?tice avant la fin de la s?ance de TP.**</font> Chacun d'entre vous doit rendre un travail personnel pour cette partie, diff?rent de celui de tous les autres ?tudiants. Les recopies et plagiats seront s?verement sanctionn?s.

Nous vous proposons ci-dessous un jeu de donn?es ? explorer avec les outils que nous avons vu en cours+TD+TP (vous pouvez utiliser un autre jeu de donn?es si vous le souhaitez---vous pouvez regarder les jeux de donn?es disponibles sur kaggle.com---mais faites attention ? ne pas perdre trop de temps ? vous le procurer, car vous devez rendre quelque chose avant la fin de la s?ance de TP).

Vous devez (i) introduire et motiver une question de votre choix (vous pouvez reprendre la question que vous aviez choisi pour le TP1, par exemple), (ii) appliquer la m?thode du bootstrap vue ci-dessus pour ?valuer l'incertitude associ?e au quantit?s estim?e pour r?pondre ? votre question et (iii) interpr?ter les r?sultats obtenus.

La grille d'?valuation suivante sera utilis?e pour noter vos travaux:

1. Motivation de l?analyse r?alis?e
    - Analyse clairement pertinente et bien justifi?e : 5
    - Motivation partiellement claire, mais manque de pr?cision : 2,5
    - Motivation incompr?hensible ou absente : 0

2. Interpr?tation des r?sultats
    - Interpr?tation correcte et pertinente des r?sultats : 5
	- Interpr?tation partiellement correcte, mais quelques lacunes : 2,5
	- Interpr?tation douteuse, incompl?te ou manquante : 0

3. Capacit? ? g?n?raliser les notions vues en cours
    - Analyse innovante sur des donn?es in?dites : 5
    - Analyse partiellement nouvelle (nouvelles donn?es ou nouvelle approche sur des donn?es vues en cours/TD/TP) : 2,5
    - Analyse quasi identique ? une vue en cours/TD/TP avec les m?mes donn?es : 0

4. Correction du code
    - Code fonctionnel et sans erreurs : 2,5
    - Code fonctionnel mais avec des erreurs partielles : 1
    - Code non fonctionnel ou contenant de nombreuses erreurs : 0

5. Propret? du code
    - Code propre, bien structur? et lisible : 2,5
    - Code lisible mais d?sordonn? ou manquant de clart? : 1
    - Code difficilement lisible et mal structur? : 0

Note totale : /20


## Jeu de donn?es propos?

In [ ]:
# Source https://www.kaggle.com/datasets/lainguyn123/student-performance-factors?resource=download
import pandas

data = pandas.read_csv("StudentPerformanceFactors.csv")
data

In [ ]:
# A vous de jouer

# Cr?dits

Ce cours est inspir? du cours data8 donn? ? UC Berkeley et en r?-utilise avec certaines modifications une partie des mat?riels (ces mat?riels sont g?n?reusement mis ? disposition publiquement sous licence Creative Commons avec attribution, consultez [https://www.data8.org](https://www.data8.org) pour plus d'informations.